# Notebook 10 — Ewaluacja, SHAP i Raport Końcowy

**Projekt:** Współzależność rynków aktywów cyfrowych i tradycyjnych systemów finansowych  

---

## Cel notebooka

Kompleksowa ewaluacja wszystkich modeli i analiza interpretowalności:

1. **Tabela porównawcza** wszystkich modeli (LSTM, GRU, XGB, LGB, Ensemble)
2. **ROC curves** i **Precision-Recall curves** na jednym wykresie
3. **Confusion matrices** — analiza błędów
4. **Krzywe kalibracji** — czy modele dobrze szacują prawdopodobieństwo?
5. **SHAP values** — które cechy najważniejsze i dlaczego?
6. **Analiza błędów** — false positives / false negatives z kontekstem
7. **Raport końcowy** — generowanie summary_report.md

In [1]:
# ============================================================
# Konfiguracja wspolna
# ============================================================
import os, warnings
os.environ.setdefault('KERAS_BACKEND', 'torch')
os.environ.setdefault('PYTORCH_ENABLE_MPS_FALLBACK', '1')
warnings.filterwarnings('ignore')
import importlib
import pandas as pd, numpy as np
import pipeline_lib as pl, models_lib as ml, run_analysis as ra
for m in (pl, ml, ra): importlib.reload(m)
pd.set_option('display.width', 160)
print(f'Ziarno losowe: {pl.SEED} | horyzont: {pl.HORIZON} sesji | embargo: {pl.EMBARGO} sesji')


Ziarno losowe: 42 | horyzont: 5 sesji | embargo: 10 sesji


In [2]:
# ============================================================
# Ewaluacja koncowa: kalibracja, krzywe oceny, interpretowalnosc
# ============================================================
oof, y, master = ra.load()
cmp = ra.model_comparison(oof, y)
ra.calibration_and_curves(oof, y)
shap_imp = ra.shap_analysis()
print('15 najwazniejszych cech wedlug sredniej bezwzglednej wartosci SHAP:')
print(shap_imp.head(15).to_string(index=False))


15 najwazniejszych cech wedlug sredniej bezwzglednej wartosci SHAP:
                Cecha  Mean |SHAP|
 wig_realized_vol_21d     2.765000
      wig_amihud_ma30     0.680620
            macro_vix     0.499353
      btc_amihud_ma30     0.438414
   cross_corr_vol_30d     0.406619
 btc_realized_vol_30d     0.318203
       macro_vix_ma30     0.314673
         btc_bb_width     0.256868
      btc_macd_signal     0.160984
    wig_parkinson_vol     0.138718
  wig_log_return_lag5     0.136063
       cross_corr_90d     0.135883
           btc_rsi_14     0.125008
btc_volume_normalized     0.120495
  btc_dist_from_ma200     0.113268


In [3]:
# ============================================================
# Raport zbiorczy
# ============================================================
res = ra.hri_analysis(oof, y, master)
ra.write_summary(cmp, res, shap_imp, oof, y)
oof.to_csv('data/processed/oof_walkforward.csv')
print('Zapisano results/summary_report.md')
print(open('results/summary_report.md', encoding='utf-8').read()[:1500])


Zapisano results/summary_report.md
# Raport zbiorczy z badania

**Data wygenerowania:** 2026-08-18 21:29

## 1. Dane

| Seria | Źródło | Uwagi |
|---|---|---|
| BTC/USD | Binance REST API (BTCUSDT, interwał dzienny) | wolumen = obrót w USDT |
| WIG20 | Biznesradar.pl (dane GPW) | wolumen = wartość obrotu w PLN |
| VIX | Cboe za pośrednictwem Yahoo Finance | |
| DXY | ICE za pośrednictwem Yahoo Finance | |
| Stopa funduszy federalnych | FRED, seria dzienna DFF | |

Pełny opis pochodzenia danych, kontroli krzyżowych i punktów kontrolnych: `DANE_POCHODZENIE.md`. Log maszynowy: `data/raw/_provenance_log.json`.

## 2. Zmienna objaśniana

Susza płynnościowa zdefiniowana jako przekroczenie przez średnie warunki płynnościowe WIG20 w kolejnych pięciu sesjach progu równego 90. percentylowi rozkładu z okresu treningowego. Wskaźnik warunków płynnościowych łączy cztery standaryzowane komponenty: niepłynność Amihuda, estymator spreadu Corwina-Schultza, niedobór obrotu względem średniej z 20 sesji or